# Wandreel Unified Pipeline Test Notebook

Test end-to-end flow in one place:
1. Extraction (`/api/metadata/extract`)
2. Intelligence (`/api/intelligence/extract`)

Use this notebook to debug each layer independently or as a chained pipeline.

## Prerequisites
- Start API server: `npm run dev:api`
- Ensure `.env` in Wandreel has OpenAI keys for intelligence sync tests


In [ ]:
import json
from urllib import request, error
import time

BASE_URL = "http://localhost:8787"
TEST_URL = "https://example.com"  # Replace with YouTube/Instagram/Web URL
EXTRACTION_MODE = "deep"           # quick | deep
INTELLIGENCE_MODE = "sync"         # sync | async


In [ ]:
def post_json(path: str, payload: dict, timeout: int = 180):
    body = json.dumps(payload).encode("utf-8")
    req = request.Request(
        f"{BASE_URL}{path}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as e:
        raw = e.read().decode("utf-8", errors="replace")
        return {"ok": False, "status": e.code, "error": raw}

def get_json(path: str, timeout: int = 180):
    req = request.Request(f"{BASE_URL}{path}", method="GET")
    try:
        with request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as e:
        raw = e.read().decode("utf-8", errors="replace")
        return {"ok": False, "status": e.code, "error": raw}


## Step 1: Run Extraction

In [ ]:
extraction = post_json("/api/metadata/extract", {"url": TEST_URL, "mode": EXTRACTION_MODE})
extraction

## Step 2A: Intelligence Sync

In [ ]:
sync_result = post_json("/api/intelligence/extract", {"source": extraction, "mode": "sync"})
sync_result

## Step 2B: Intelligence Async

In [ ]:
async_create = post_json("/api/intelligence/extract", {"source": extraction, "mode": "async"})
async_create

In [ ]:
job_result = None
job_id = async_create.get("jobId")
if job_id:
    for _ in range(20):
        candidate = get_json(f"/api/intelligence/jobs/{job_id}")
        status = (candidate.get("job") or {}).get("status")
        if status in {"completed", "failed"}:
            job_result = candidate
            break
        time.sleep(1)
job_result

## Layer-Specific Quick Views

In [ ]:
meta = extraction.get("metadata", {})
print("TITLE:", meta.get("title"))
print("DESCRIPTION:", meta.get("description"))
print("PLATFORM:", meta.get("platform"))
print("PROVIDER:", meta.get("provider"))

transcript = extraction.get("transcript") or {}
ocr = extraction.get("ocr") or {}
print("TRANSCRIPT USED:", transcript.get("used"), "SOURCE:", transcript.get("source"))
print("OCR USED:", ocr.get("used"))

In [ ]:
intel_payload = sync_result.get("output", {}) if isinstance(sync_result, dict) else {}
print("STATUS:", intel_payload.get("status"))
print("CATEGORIES:", intel_payload.get("categoriesPresent"))
print("WEAK MENTIONS:", intel_payload.get("weakMentions"))
print("ENTITY COUNT:", len(intel_payload.get("entities", [])))


## Batch Links (Extraction -> Intelligence Sync)

In [ ]:
urls = [
    TEST_URL,
    # "https://www.youtube.com/watch?v=...",
    # "https://www.instagram.com/reel/...",
]

rows = []
for u in urls:
    ex = post_json("/api/metadata/extract", {"url": u, "mode": EXTRACTION_MODE})
    intel = post_json("/api/intelligence/extract", {"source": ex, "mode": "sync"})
    out = intel.get("output", {}) if isinstance(intel, dict) else {}
    rows.append({
        "url": u,
        "platform": (ex.get("metadata") or {}).get("platform"),
        "title": (ex.get("metadata") or {}).get("title"),
        "status": out.get("status"),
        "categoriesPresent": out.get("categoriesPresent"),
        "entityCount": len(out.get("entities", [])),
    })

rows